<a href="https://colab.research.google.com/github/NeuronEdge67/Hands-On-Machine-Learning-...By-Aur-lien-G-ron-/blob/main/10_neural_nets_with_pytorch.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [2]:
import torch

In [3]:
torch.set_default_device('cuda')

In [4]:
X = torch.tensor([[1.0, 4.0, 7.0], [2.0, 3.0, 6.0]])
X

tensor([[1., 4., 7.],
        [2., 3., 6.]], device='cuda:0')

In [5]:
X.shape

torch.Size([2, 3])

In [6]:
X.dtype

torch.float32

In [7]:
a = X.device
a

device(type='cuda', index=0)

In [8]:
x = torch.tensor(6.0, requires_grad=True)
f = x ** 2
f

tensor(36., device='cuda:0', grad_fn=<PowBackward0>)

In [9]:
f.backward(retain_graph=True)
x.grad

tensor(12., device='cuda:0')

In [10]:
lr = 0.1
x = torch.tensor(5.0, requires_grad=True)
for iter in range(100):
  f = x**2
  f.backward()
  with torch.no_grad():
    x -= lr * x.grad
  x.grad.zero_()

In [11]:
x

tensor(1.0185e-09, device='cuda:0', requires_grad=True)

In [12]:
from sklearn.datasets import fetch_california_housing
from sklearn.metrics import root_mean_squared_error
from sklearn.model_selection import train_test_split
from sklearn.neural_network import MLPRegressor
from sklearn.pipeline import make_pipeline
from sklearn.preprocessing import StandardScaler

In [13]:
housing = fetch_california_housing()
X_train, X_temp, y_train, y_temp = train_test_split(
    housing.data, housing.target, test_size=0.2, random_state=42)
X_valid, X_test, y_valid, y_test = train_test_split(
    X_temp, y_temp, test_size=0.5, random_state=42)

In [14]:
X_train = torch.FloatTensor(X_train)
X_valid = torch.FloatTensor(X_valid)
X_test = torch.FloatTensor(X_test)
means = X_train.mean(dim=0, keepdims=True)
stds = X_train.std(dim=0, keepdims=True)
X_train = (X_train - means) / stds
X_valid = (X_valid - means) / stds
X_test = (X_test - means) / stds

In [15]:
y_train = torch.FloatTensor(y_train).view(-1, 1)
y_valid = torch.FloatTensor(y_valid).view(-1, 1)
y_test = torch.FloatTensor(y_test).view(-1, 1)

In [16]:
torch.manual_seed(42)
n_features = X_train.shape[1]
w = torch.randn((n_features, 1), requires_grad=True)
b = torch.tensor(0., requires_grad=True)

In [17]:
learning_rate = 0.4
n_epochs = 20

X_train = X_train.to('cuda')
y_train = y_train.to('cuda')
w = w.to('cuda')
b = b.to('cuda')

for epoch in range(n_epochs):
    y_pred = X_train @ w + b
    loss = ((y_pred - y_train) ** 2).mean()
    loss.backward()
    with torch.no_grad():
        b -= learning_rate * b.grad
        w -= learning_rate * w.grad
        b.grad.zero_()
        w.grad.zero_()
    print(f"Epoch {epoch + 1}/{n_epochs}, Loss: {loss.item()}")

Epoch 1/20, Loss: 16.47666358947754
Epoch 2/20, Loss: 1.2107000350952148
Epoch 3/20, Loss: 0.6819401383399963
Epoch 4/20, Loss: 0.5730571746826172
Epoch 5/20, Loss: 0.5387604236602783
Epoch 6/20, Loss: 0.5267452001571655
Epoch 7/20, Loss: 0.5223101377487183
Epoch 8/20, Loss: 0.5205764174461365
Epoch 9/20, Loss: 0.5198367834091187
Epoch 10/20, Loss: 0.5194748640060425
Epoch 11/20, Loss: 0.519262969493866
Epoch 12/20, Loss: 0.519115149974823
Epoch 13/20, Loss: 0.5189981460571289
Epoch 14/20, Loss: 0.5188987851142883
Epoch 15/20, Loss: 0.5188113451004028
Epoch 16/20, Loss: 0.5187333226203918
Epoch 17/20, Loss: 0.5186628699302673
Epoch 18/20, Loss: 0.518599271774292
Epoch 19/20, Loss: 0.5185415744781494
Epoch 20/20, Loss: 0.5184893012046814


In [18]:
import torch.nn as nn

torch.manual_seed(42)
model = nn.Linear(in_features=n_features, out_features=1)

In [19]:
print(model.bias)
print(model.weight)

Parameter containing:
tensor([0.3449], device='cuda:0', requires_grad=True)
Parameter containing:
tensor([[ 0.0799, -0.3464, -0.0718, -0.3251, -0.2431, -0.0124,  0.1671, -0.0665]],
       device='cuda:0', requires_grad=True)


In [20]:
model(X_train[:2])

tensor([[-0.2229],
        [-0.3159]], device='cuda:0', grad_fn=<AddmmBackward0>)

In [21]:
def train_bgd(model, optimizer, criterion, X_train, y_train, n_epochs):
 for epoch in range(n_epochs):
  y_pred = model(X_train)
  loss = criterion(y_pred, y_train)
  loss.backward()
  optimizer.step()
  optimizer.zero_grad()
  print(f"Epoch {epoch + 1}/{n_epochs}, Loss: {loss.item()}")

In [22]:
torch.manual_seed(42)
model = nn.Sequential(
    nn.Linear(n_features, 50),
    nn.ReLU(),
    nn.Linear(50, 40),
    nn.ReLU(),
    nn.Linear(40, 1)
)

In [27]:
learning_rate = 0.01
optimizer = torch.optim.SGD(model.parameters(), lr=learning_rate)
mse = nn.MSELoss()
train_bgd(model, optimizer, mse, X_train, y_train, n_epochs)

Epoch 1/20, Loss: 0.6348775029182434
Epoch 2/20, Loss: 0.6098587512969971
Epoch 3/20, Loss: 0.5897341966629028
Epoch 4/20, Loss: 0.5734017491340637
Epoch 5/20, Loss: 0.5600319504737854
Epoch 6/20, Loss: 0.5490034222602844
Epoch 7/20, Loss: 0.5398461222648621
Epoch 8/20, Loss: 0.5321906208992004
Epoch 9/20, Loss: 0.5257459282875061
Epoch 10/20, Loss: 0.520285427570343
Epoch 11/20, Loss: 0.5156362056732178
Epoch 12/20, Loss: 0.5116581320762634
Epoch 13/20, Loss: 0.5082327723503113
Epoch 14/20, Loss: 0.505268394947052
Epoch 15/20, Loss: 0.5026864409446716
Epoch 16/20, Loss: 0.5004231333732605
Epoch 17/20, Loss: 0.49842965602874756
Epoch 18/20, Loss: 0.4966643452644348
Epoch 19/20, Loss: 0.4950939416885376
Epoch 20/20, Loss: 0.4936862289905548
